In [2]:
import pandas as pd

from src.thetadata_pipeline.ib.lattency import trades_collector
from src.thetadata_pipeline.pipeline_config import load_pipeline_config
from src.thetadata_pipeline.settings import get_settings

cfg = load_pipeline_config()
settings = get_settings()

In [3]:
strangle_trades = pd.read_parquet(get_settings().strategies_dir / 'SPY_strangle_trades.parquet')

ib_files = list(settings.ib_states_dir.glob(f"{cfg.analysis.account_id}*.csv"))
ib_trades = trades_collector(cfg.analysis.account_id, settings)

In [190]:
important_cols = [
    'Symbol', 'Date/Time', 'T. Price', 'Proceeds', 'Comm/Fee', 'Realized P/L',
    'ticker', 'expiration', 'strike', 'right', 'trade_dt', 'trade_date', 'trade_ms', 'quantity'
]
ib_trades = ib_trades[important_cols].copy()

for col in ['T. Price', 'Proceeds', 'Comm/Fee', 'Realized P/L', 'strike', 'trade_ms', 'quantity']:
    ib_trades[col] = pd.to_numeric(ib_trades[col], errors='coerce')
ib_trades['right'] = ib_trades['right'].str.lower()

leg_keys = ['ticker', 'expiration', 'strike', 'right']


def summarize_side(rows: pd.DataFrame, prefix: str) -> pd.DataFrame:
    return (
        rows.sort_values('trade_dt')
        .groupby(leg_keys, as_index=False)
        .agg(
            **{
                f'{prefix}_Symbol': ('Symbol', 'first'),
                f'{prefix}_fill_price': ('T. Price', 'first'),
                f'{prefix}_proceeds': ('Proceeds', 'sum'),
                f'{prefix}_comm_fee': ('Comm/Fee', 'sum'),
                f'{prefix}_realized_pl': ('Realized P/L', 'sum'),
                f'{prefix}_trade_dt': ('trade_dt', 'first'),
                f'{prefix}_trade_date': ('trade_date', 'first'),
                f'{prefix}_trade_ms': ('trade_ms', 'first'),
                f'{prefix}_quantity': ('quantity', 'sum'),
            }
        )
    )


entries = summarize_side(ib_trades[ib_trades['quantity'] < 0], 'ent')
exits = summarize_side(ib_trades[ib_trades['quantity'] > 0], 'ext')

ib_legs = entries.merge(exits, on=leg_keys, how='left')
ib_legs['contracts'] = ib_legs['ent_quantity'].abs()
ib_legs['leg_comm_fee'] = ib_legs['ent_comm_fee'].fillna(0) + ib_legs['ext_comm_fee'].fillna(0)
ib_legs['leg_profit_cash'] = ib_legs['ent_proceeds'].fillna(0) + ib_legs['ext_proceeds'].fillna(0)
ib_legs['leg_profit'] = ib_legs['leg_profit_cash'] / ib_legs['contracts'] / 100

join_cols = ['ticker', 'expiration', 'ent_trade_date']
call = ib_legs[ib_legs['right'] == 'c'].drop(columns='right')
put = ib_legs[ib_legs['right'] == 'p'].drop(columns='right')
call = call.rename(columns={c: f'call_{c}' for c in call.columns if c not in join_cols})
put = put.rename(columns={c: f'put_{c}' for c in put.columns if c not in join_cols})

ib_strangles = call.merge(put, on=join_cols, how='outer')
ib_strangles['ib_strangle_profit_cash'] = ib_strangles['call_leg_profit_cash'] + ib_strangles['put_leg_profit_cash']
ib_strangles['ib_strangle_profit'] = ib_strangles['call_leg_profit'] + ib_strangles['put_leg_profit']
ib_strangles['ib_comm_fee'] = ib_strangles['call_leg_comm_fee'] + ib_strangles['put_leg_comm_fee']

ib_strangles

,ticker,expiration,call_strike,call_ent_Symbol,call_ent_fill_price,call_ent_proceeds,call_ent_comm_fee,call_ent_realized_pl,call_ent_trade_dt,ent_trade_date,...,put_ext_trade_date,put_ext_trade_ms,put_ext_quantity,put_contracts,put_leg_comm_fee,put_leg_profit_cash,put_leg_profit,ib_strangle_profit_cash,ib_strangle_profit,ib_comm_fee
0,AAPL,2024-11-22,227.5,AAPL 22NOV24 227.5 C,2.98,298.0,-1.062624,0.0,2024-11-19 14:34:02,2024-11-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,2024-11-22,NaN,NaN,NaN,NaN,NaN,NaN,NaT,2024-11-22,...,2024-11-19,52442000.0,1.0,1.0,-1.051550,-122.0,-1.22,NaN,NaN,NaN
2,SPY,2024-12-18,605.0,SPY 18DEC24 605 C,1.07,107.0,-1.539715,0.0,2024-12-18 09:37:55,2024-12-18,...,2024-12-18,50825000.0,1.0,1.0,-2.583387,-48.0,-0.48,-98.0,-0.98,-5.169551
3,SPY,2024-12-19,595.0,SPY 19DEC24 595 C,0.99,99.0,-1.539492,0.0,2024-12-19 09:36:16,2024-12-19,...,2024-12-19,34671000.0,1.0,1.0,-2.574554,-82.0,-0.82,17.0,0.17,-4.114046
4,SPY,2024-12-20,584.0,SPY 20DEC24 584 C,1.47,147.0,-1.050827,0.0,2024-12-20 09:36:28,2024-12-20,...,2024-12-20,58800000.0,1.0,1.0,-1.050354,130.0,1.30,56.0,0.56,-3.135131
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
338,SPY,2026-05-12,737.0,SPY 12MAY26 737 C,1.01,404.0,-2.774482,0.0,2026-05-12 09:36:07,2026-05-12,...,2026-05-12,37878000.0,4.0,4.0,-4.086658,-204.0,-0.51,-12.0,-0.03,-9.594141
339,SPY,2026-05-13,739.0,SPY 13MAY26 739 C,0.67,268.0,-2.571681,0.0,2026-05-13 09:36:07,2026-05-13,...,2026-05-13,35277000.0,4.0,4.0,-4.277318,-248.0,-0.62,-412.0,-1.03,-8.881998
340,SPY,2026-05-14,746.0,SPY 14MAY26 746 C,0.91,364.0,-2.053658,0.0,2026-05-14 09:36:22,2026-05-14,...,2026-05-14,58320000.0,4.0,4.0,-4.557235,384.0,0.96,188.0,0.47,-9.403894
341,SPY,2026-05-15,742.0,SPY 15MAY26 742 C,1.22,488.0,-4.136213,0.0,2026-05-15 09:36:08,2026-05-15,...,2026-05-15,35236000.0,4.0,4.0,-4.090861,-300.0,-0.75,-588.0,-1.47,-10.980074


# Nan

In [191]:
mask = ib_strangles.isna().sum(axis=1) > 0
ib_strangles[mask]

,ticker,expiration,call_strike,call_ent_Symbol,call_ent_fill_price,call_ent_proceeds,call_ent_comm_fee,call_ent_realized_pl,call_ent_trade_dt,ent_trade_date,...,put_ext_trade_date,put_ext_trade_ms,put_ext_quantity,put_contracts,put_leg_comm_fee,put_leg_profit_cash,put_leg_profit,ib_strangle_profit_cash,ib_strangle_profit,ib_comm_fee
0,AAPL,2024-11-22,227.5,AAPL 22NOV24 227.5 C,2.98,298.0,-1.062624,0.0,2024-11-19 14:34:02,2024-11-19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AAPL,2024-11-22,NaN,NaN,NaN,NaN,NaN,NaN,NaT,2024-11-22,...,2024-11-19,52442000.0,1.0,1.0,-1.05155,-122.0,-1.22,NaN,NaN,NaN


# Join back and real

In [192]:
ib_strangles = ib_strangles[~mask]

ib_by_days = (
    ib_strangles.groupby('ent_trade_date').agg(
        ib_strangle_profit=('ib_strangle_profit', 'sum'),
        ib_comm_fee=('ib_comm_fee', 'sum'),
        ib_call_strike=('call_strike', 'first'),
        ib_call_enter_ms=('call_ent_trade_ms', 'first'),
        ib_call_exit_ms=('call_ext_trade_ms', 'first'),
        ib_put_strike=('put_strike', 'first'),
        ib_put_enter_ms=('put_ent_trade_ms', 'first'),
        ib_put_exit_ms=('put_ext_trade_ms', 'first'),
    )
)
backtest_by_days = (
    strangle_trades[strangle_trades['date'] >= ib_by_days.index[0]]
    .groupby('date').agg(
        back_strangle_profit=('strangle_profit', 'sum'),
        back_call_strike=('strike', 'first'),
        back_call_enter_ms=('ent_time_ms', 'first'),
        back_call_exit_ms=('ext_time_ms', 'first'),
        back_put_strike=('strike_put', 'first'),
        back_put_enter_ms=('ent_time_ms_put', 'first'),
        back_put_exit_ms=('ext_time_ms_put', 'first'),
    )
)
both_by_days = pd.merge(ib_by_days, backtest_by_days, left_index=True, right_index=True, how='outer')
both_by_days = both_by_days[~both_by_days['back_strangle_profit'].isna()]

# недобор $566
print(both_by_days['ib_strangle_profit'].sum() - both_by_days['back_strangle_profit'].sum())

-5.657666666666671


# Days without trades

In [173]:
# $294 из-за дней, когда торги были пропущены
mask = both_by_days['ib_strangle_profit'].isna()
print(both_by_days[mask]['back_strangle_profit'].sum(), len(both_by_days[mask]))

both_by_days = both_by_days[~mask]
both_by_days['diff'] = both_by_days['ib_strangle_profit'] - both_by_days['back_strangle_profit']

2.94 13


# Days with lag in open time

In [174]:
both_base_strikes = both_by_days[
    (both_by_days['ib_call_strike'] == both_by_days['back_call_strike'])
    & (both_by_days['ib_put_strike'] == both_by_days['back_put_strike'])
]

both_base_strikes['ent_call_ms_diff'] = (both_base_strikes['ib_call_enter_ms'] - both_base_strikes[
    'back_call_enter_ms']) / 1000
both_base_strikes['ext_call_ms_diff'] = (both_base_strikes['ib_call_exit_ms'] - both_base_strikes[
    'back_call_exit_ms']) / 1000
both_base_strikes['ent_put_ms_diff'] = (both_base_strikes['ib_put_enter_ms'] - both_base_strikes[
    'back_put_enter_ms']) / 1000
both_base_strikes['ext_put_ms_diff'] = (both_base_strikes['ib_put_exit_ms'] - both_base_strikes[
    'back_put_exit_ms']) / 1000

In [175]:
# $150 из-за того, что входил позже 9:36:10
mask = ((both_base_strikes['ent_call_ms_diff'].abs() > 10) | (both_base_strikes['ent_put_ms_diff'].abs() > 10))
print(both_base_strikes[mask]['diff'].sum(), len(both_base_strikes[mask]))

both_by_days = both_by_days[~both_by_days.index.isin(both_base_strikes[mask].index)]
both_base_strikes = both_base_strikes[~mask]

-1.4983333333333317 103


In [179]:
# $160 из-за того, что бект-тест ищёт выход по stock quotes, а не stock trades. А так же, потому что иногда бек-тест совершает выход по фантомным stock quotes.
mask = ((both_base_strikes['ext_call_ms_diff'].abs() > 10) | (both_base_strikes['ext_put_ms_diff'].abs() > 10))
print(both_base_strikes[mask]['diff'].sum(), len(both_base_strikes[mask]))

both_by_days = both_by_days[~both_by_days.index.isin(both_base_strikes.index)]

-1.6008333333333342 32


# Different strikes

In [183]:
# $11.5 из-за разницы в страйках, так как иногда страйки находятся на однинаковом расстоянии от 0.35 дельты и какой будет торговаться решает букавально секунда.
diff_strikes_len = len(
    both_by_days[(both_by_days['ib_call_strike'] != both_by_days['back_call_strike'])
    | (both_by_days['ib_put_strike'] != both_by_days['back_put_strike'])]
)
diff_strikes_len == len(both_by_days), len(both_by_days), both_by_days['diff'].sum()

(True, 109, np.float64(-0.1151666666666673))

In [184]:
# 566 - 294 - 150 - 160 - 11.5 = $49.5 - моё преимущество, что когда разница между бектестом и реалом менее 10 секунд. Я получаю цены выгоднее, чем видит бектест.

73.5